# MediTrack: Data Preparation & Statistical Hypothesis Testing
### Project CP-02: Readmission Risk Prediction and Clinical Decision Support
This notebook executes the ingestion, clinical exclusions, missing data handling, and hypothesis tests required by Section 5 of the project brief.

In [1]:

import pandas as pd
import numpy as np
from scipy import stats
import statsmodels.api as sm
from src.data_prep import clean_and_prepare_data

# 1. Inspect Raw Data & Run Preprocessing Pipeline
raw_path = 'data/raw/diabetic_data.csv'
df_cleaned = clean_and_prepare_data(raw_path, 'data/processed')
print(f"Cleaned dataset records: {len(df_cleaned):,}")
df_cleaned[['encounter_id', 'patient_nbr', 'age_group', 'admission_type_name', 'time_in_hospital', 'diag_1_category', 'readmitted_30d']].head()


Loading raw dataset from data/raw/diabetic_data.csv...


Initial encounters loaded: 101766
Excluded 2423 deceased/hospice records. Remaining: 99343


Cleaned dataset saved to data/processed/cleaned_encounters.csv (Records: 99340)
Cleaned dataset records: 99,340


,encounter_id,patient_nbr,age_group,admission_type_name,time_in_hospital,diag_1_category,readmitted_30d
0,2278392,8222157,[0-10),NULL,1,Diabetes,0
1,149190,55629189,[10-20),Emergency,3,Other,0
2,64410,86047875,[20-30),Emergency,2,Other,0
3,500364,82442376,[30-40),Emergency,2,Other,0
4,16680,42519267,[40-50),Emergency,1,Neoplasms,0


In [2]:

# 2. Base Rate of 30-Day Readmission
total = len(df_cleaned)
readm_30d = (df_cleaned['readmitted_30d'] == 1).sum()
base_rate = readm_30d / total
print(f"Total encounters: {total:,}")
print(f"30-day Readmissions: {readm_30d:,} ({base_rate*100:.2f}%)")

# Breakdown by original category
df_cleaned['readmitted'].value_counts()


Total encounters: 99,340
30-day Readmissions: 11,314 (11.39%)


readmitted
NO     52524
>30    35502
<30    11314
Name: count, dtype: int64

In [3]:

# 3. Hypothesis Test 1: Age Group Differences (Chi-Square)
age_ct = pd.crosstab(df_cleaned['age_group'], df_cleaned['readmitted_30d'])
chi2, p_val, dof, _ = stats.chi2_contingency(age_ct)
print(f"Chi-Square Statistic: {chi2:.3f}, df: {dof}, p-value: {p_val:.4e}")

age_rates = df_cleaned.groupby('age_group')['readmitted_30d'].mean() * 100
print("\nReadmission Rate by Age Group (%):")
print(age_rates)


Chi-Square Statistic: 133.625, df: 9, p-value: 2.1286e-24

Readmission Rate by Age Group (%):
age_group
[0-10)       1.875000
[10-20)      5.797101
[20-30)     14.311704
[30-40)     11.264612
[40-50)     10.658895
[50-60)      9.771395
[60-70)     11.302022
[70-80)     12.057326
[80-90)     12.565413
[90-100)    11.896485
Name: readmitted_30d, dtype: float64


In [4]:

# 4. Hypothesis Test 2: Admission Type (Chi-Square & Odds Ratio)
adm_ct = pd.crosstab(df_cleaned['admission_type_name'], df_cleaned['readmitted_30d'])
chi2_adm, p_adm, dof_adm, _ = stats.chi2_contingency(adm_ct)
print(f"Admission Type Chi2: {chi2_adm:.3f}, df: {dof_adm}, p-value: {p_adm:.4e}")

# Odds Ratio Emergency vs Elective
em_pos = ((df_cleaned['admission_type_name'] == 'Emergency') & (df_cleaned['readmitted_30d'] == 1)).sum()
em_neg = ((df_cleaned['admission_type_name'] == 'Emergency') & (df_cleaned['readmitted_30d'] == 0)).sum()
el_pos = ((df_cleaned['admission_type_name'] == 'Elective') & (df_cleaned['readmitted_30d'] == 1)).sum()
el_neg = ((df_cleaned['admission_type_name'] == 'Elective') & (df_cleaned['readmitted_30d'] == 0)).sum()
odds_ratio = (em_pos / em_neg) / (el_pos / el_neg)
print(f"Odds Ratio (Emergency vs Elective): {odds_ratio:.3f}")


Admission Type Chi2: 32.588, df: 7, p-value: 3.1586e-05
Odds Ratio (Emergency vs Elective): 1.145


In [5]:

# 5. Hypothesis Test 3: Prior Inpatient History (t-test, Mann-Whitney U, Logit Odds Ratio)
readm_inp = df_cleaned[df_cleaned['readmitted_30d'] == 1]['number_inpatient']
noreadm_inp = df_cleaned[df_cleaned['readmitted_30d'] == 0]['number_inpatient']

t_stat, t_pval = stats.ttest_ind(readm_inp, noreadm_inp, equal_var=False)
u_stat, u_pval = stats.mannwhitneyu(readm_inp, noreadm_inp)
print(f"Mean Prior Inpatient (Readmitted): {readm_inp.mean():.3f}")
print(f"Mean Prior Inpatient (Non-Readmitted): {noreadm_inp.mean():.3f}")
print(f"Welch t-stat: {t_stat:.3f}, p-val: {t_pval:.4e}")
print(f"Mann-Whitney U: {u_stat:.1f}, p-val: {u_pval:.4e}")

# Logistic Odds Ratio
X = sm.add_constant(df_cleaned['number_inpatient'])
logit_mod = sm.Logit(df_cleaned['readmitted_30d'], X).fit(disp=False)
print(f"Logistic Odds Ratio per prior stay: {np.exp(logit_mod.params['number_inpatient']):.3f}")


Mean Prior Inpatient (Readmitted): 1.223
Mean Prior Inpatient (Non-Readmitted): 0.555
Welch t-stat: 35.584, p-val: 4.8684e-264
Mann-Whitney U: 604789932.5, p-val: 0.0000e+00
Logistic Odds Ratio per prior stay: 1.337
